In [2]:
from __future__ import annotations
from pathlib import Path
from typing import Iterable, Union
import requests


def sdss_id_to_astra_url(sdss_id: Union[int, str],version: str = "0.8.1",
    base_url: str = "https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.8.1/spectra/star") -> str:
    """
    Build the Astra mwmStar FITS URL from an SDSS ID.

    Example:
        102591990 -> .../19/90/mwmStar-0.8.1-102591990.fits
    """
    sid = str(sdss_id).strip()

    if not sid.isdigit():
        raise ValueError(f"sdss_id must be numeric, got: {sdss_id!r}")

    last4 = sid[-4:].zfill(4)
    sub1 = last4[:2]
    sub2 = last4[2:]

    # Keep version in sync with the base path if you change it.
    base_url = base_url.rstrip("/")
    return f"{base_url}/{sub1}/{sub2}/mwmStar-{version}-{sid}.fits"


def download_astra_spectra(sdss_ids: Iterable[Union[int, str]],out_dir: Union[str, Path],
    version: str = "0.8.1",base_url: str = "https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.8.1/spectra/star",
    timeout: int = 60,overwrite: bool = False,auth: tuple[str, str] | None = None) -> list[dict]:
    """
    Download Astra mwmStar FITS files for a list of SDSS IDs.

    Parameters
    ----------
    sdss_ids : iterable
        List or iterable of SDSS IDs.
    out_dir : str or Path
        Output directory where files will be saved.
    version : str
        Astra version used in the filename.
    base_url : str
        Base archive URL up to .../spectra/star
    timeout : int
        Request timeout in seconds.
    overwrite : bool
        Whether to overwrite existing files.
    auth : tuple(username, password) or None
        Optional HTTP auth if the archive requires credentials.

    Returns
    -------
    results : list of dict
        Per-ID download status.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    session = requests.Session()

    results: list[dict] = []

    for sdss_id in sdss_ids:
        sid = str(sdss_id).strip()
        try:
            url = sdss_id_to_astra_url(sid,version=version,base_url=base_url)
        except ValueError as e:
            results.append({"sdss_id": sid,"status": "invalid_id","error": str(e)})
            continue

        filename = f"mwmStar-{version}-{sid}.fits"
        dest = out_dir / filename

        if dest.exists() and not overwrite:
            results.append({"sdss_id": sid,"status": "exists","path": str(dest),"url": url})
            continue

        try:
            with session.get(url, stream=True, timeout=timeout, auth=auth) as r:
                if r.status_code != 200:
                    results.append({"sdss_id": sid,"status": "failed","http_status": r.status_code,
                            "url": url})
                    continue

                with open(dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            results.append({"sdss_id": sid,"status": "downloaded","path": str(dest),"url": url})

        except requests.RequestException as e:
            results.append({"sdss_id": sid, "status": "error","error": str(e),"url": url})

    return results

In [6]:
sdss_ids = [91050455]

results = download_astra_spectra(sdss_ids,
    out_dir="/Users/asamantaray/Downloads/",
    version="0.8.1",
    base_url="https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.8.1/spectra/star",
    overwrite=True, auth=None)  # put ("username", "password") here if needed

for r in results:
    print(r)

{'sdss_id': '91050455', 'status': 'downloaded', 'path': '/Users/asamantaray/Downloads/mwmStar-0.8.1-91050455.fits', 'url': 'https://data.sdss5.org/sas/sdsswork/mwm/spectro/astra/0.8.1/spectra/star/04/55/mwmStar-0.8.1-91050455.fits'}
